# Voice feature grouped ablation — reuse prior search

This notebook does **not** rerun the layer scan, train an SAE, rank features, or repeat the 20 individual ablations. It loads the previously downloaded `voice_feature_search_05b.zip`, reuses its layer-19 activations, SAE, and feature screen, and runs only fixed-CoT grouped interventions.

Upload the prior result ZIP and the paired-voice adapter when prompted.

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU"
print(torch.cuda.get_device_name(0))

REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"
!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!pip install -q peft "transformers<4.50" accelerate matplotlib
!pip uninstall -y torchao >/dev/null 2>&1

In [ ]:
from pathlib import Path
from google.colab import files
from transformers import AutoTokenizer
import hashlib, json, shutil

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
VOICE_ADAPTER = Path("checkpoints/qwen05b-cot-sft-voice-paired")
RESULTS = Path("/content/voice_feature_search_05b")
OUT = Path("/content/voice_feature_group_ablation_05b")
OUT.mkdir(parents=True, exist_ok=True)

print("Upload only voice_feature_search_05b.zip")
zip_upload = files.upload()
zip_names = [name for name in zip_upload if name.endswith(".zip")]
assert len(zip_names) == 1, f"Expected one search-results ZIP, got {zip_names}"
search_zip = Path("/content") / Path(zip_names[0]).name
search_zip.write_bytes(zip_upload[zip_names[0]])
shutil.unpack_archive(search_zip, RESULTS)
print("Prior artifacts:", sorted(path.name for path in RESULTS.iterdir()))

In [ ]:
print("Now upload these two files together from checkpoints/qwen05b-cot-sft-voice-minimal (3):")
print("  adapter_config.json")
print("  adapter_model.safetensors")
adapter_upload = files.upload()

VOICE_ADAPTER.mkdir(parents=True, exist_ok=True)
for name, data in adapter_upload.items():
    basename = Path(name).name
    if basename in {"adapter_config.json", "adapter_model.safetensors"}:
        (VOICE_ADAPTER / basename).write_bytes(data)
assert (VOICE_ADAPTER / "adapter_config.json").is_file()
assert (VOICE_ADAPTER / "adapter_model.safetensors").is_file()
AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(VOICE_ADAPTER)
VOICE_SHA256 = hashlib.sha256((VOICE_ADAPTER / "adapter_model.safetensors").read_bytes()).hexdigest()
print("Voice adapter SHA256:", VOICE_SHA256)
print("The next cell verifies this hash against the completed search.")

In [ ]:
import torch.nn.functional as F
from sparse_autoencoders.run_sae import load_model, transformer_layers
from sparse_autoencoders.sae import SparseAutoencoder
from evaluation.evaluate_ethics_morality import build_prompt

DEVICE = torch.device("cuda")
BATCH_SIZE = 16
PAIRS_PATH = Path("data/validation_data/synthetic_ethics_voice_paired_val.jsonl")

metadata = json.loads((RESULTS / "experiment.json").read_text())
assert "VOICE_SHA256" in globals(), "Run the adapter-upload cell immediately above this cell first"
assert metadata["voice_adapter_sha256"] == VOICE_SHA256, "Uploaded adapter does not match the completed search"
BEST_LAYER = int(metadata["best_layer"])
assert BEST_LAYER == 19, f"Expected the saved layer-19 search, found layer {BEST_LAYER}"
activations = torch.load(RESULTS / "all_layer_activations.pt", map_location="cpu", weights_only=True)
feature_effects = json.loads((RESULTS / "single_feature_ablation.json").read_text())
feature_effects.sort(key=lambda row: row["logit_recovery_toward_base"], reverse=True)

sae_checkpoint = torch.load(RESULTS / f"sae_l{BEST_LAYER}" / "sae.pt", map_location="cpu", weights_only=True)
sae = SparseAutoencoder(896, int(sae_checkpoint["dict_size"]))
sae.load_state_dict(sae_checkpoint["state_dict"])
sae.to(DEVICE).eval()

positive_features = [row["feature"] for row in feature_effects if row["logit_recovery_toward_base"] > 0]
assert len(positive_features) >= 10
FEATURE_GROUPS = {
    "causal_top1": positive_features[:1],
    "causal_top3": positive_features[:3],
    "causal_top6": positive_features[:6],
    "causal_top10": positive_features[:10],
    "anti_recovery_control6": [row["feature"] for row in feature_effects[-6:]],
}
print("Best layer:", BEST_LAYER)
print("Grouped arms:", FEATURE_GROUPS)

rows = [json.loads(line) for line in PAIRS_PATH.read_text().splitlines() if line.strip()]
groups = {}
for row in rows:
    groups.setdefault(int(row["pair_index"]), {})[row["voice"]] = row
assert len(groups) == 100 and all(set(pair) == {"active", "passive"} for pair in groups.values())
pairs = [(index, pair["active"], pair["passive"]) for index, pair in sorted(groups.items())]

def readout_text(row):
    return f"{build_prompt(row)} {row['chain_of_thought']}\nFinal answer:"

validation_active_texts = [readout_text(active) for _, active, _ in pairs[50:]]
validation_passive_texts = [readout_text(passive) for _, _, passive in pairs[50:]]
voice_tokenizer, voice_model = load_model(str(VOICE_ADAPTER), DEVICE)
print("Loaded cached search data and voice model; no search was rerun.")

In [ ]:
def label_token_ids(tokenizer):
    ids = {}
    for label in ("0", "1"):
        encoded = tokenizer(label, add_special_tokens=False)["input_ids"]
        assert len(encoded) == 1, (label, encoded)
        ids[label] = encoded[0]
    return ids

@torch.no_grad()
def evaluate_intervention(texts, *, direction=None, center=None, features=None):
    assert (direction is not None) != (features is not None)
    voice_tokenizer.padding_side = "right"
    positions = None
    zero_id, one_id = label_token_ids(voice_tokenizer)["0"], label_token_ids(voice_tokenizer)["1"]
    if direction is not None:
        direction = direction.to(DEVICE)
        center = center.to(DEVICE)
    else:
        feature_ids = torch.tensor(features, device=DEVICE, dtype=torch.long)
        decoder = sae.decoder.weight.index_select(1, feature_ids)

    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        index = torch.arange(hidden.shape[0], device=hidden.device)
        target = hidden[index, positions].float()
        if direction is not None:
            coefficient = ((target - center) * direction).sum(-1, keepdim=True)
            patched_target = target - coefficient * direction
        else:
            selected = sae.encode(target).index_select(1, feature_ids)
            patched_target = target - selected @ decoder.T
        patched = hidden.clone()
        patched[index, positions] = patched_target.to(hidden.dtype)
        return (patched,) + output[1:] if isinstance(output, tuple) else patched

    handle = transformer_layers(voice_model)[BEST_LAYER].register_forward_hook(hook)
    margins = []
    try:
        for start in range(0, len(texts), BATCH_SIZE):
            inputs = voice_tokenizer(
                texts[start:start + BATCH_SIZE], return_tensors="pt", padding=True,
                truncation=True, max_length=1024,
            ).to(DEVICE)
            positions = inputs["attention_mask"].sum(1) - 1
            output = voice_model(**inputs, use_cache=False)
            index = torch.arange(positions.shape[0], device=DEVICE)
            logits = output.logits[index, positions]
            margins.append((logits[:, one_id] - logits[:, zero_id]).float().cpu())
    finally:
        handle.remove()
    return torch.cat(margins)

In [ ]:
DISCOVERY = slice(0, 50)
VALIDATION = slice(50, 100)
base_active_margin = activations["base_active_margin"]
base_passive_margin = activations["base_passive_margin"]
voice_active_margin = activations["voice_active_margin"]
voice_passive_margin = activations["voice_passive_margin"]
base_validation = torch.cat([base_active_margin[VALIDATION], base_passive_margin[VALIDATION]])
voice_validation = torch.cat([voice_active_margin[VALIDATION], voice_passive_margin[VALIDATION]])
base_gap = float(base_active_margin[VALIDATION].mean() - base_passive_margin[VALIDATION].mean())
voice_gap = float(voice_active_margin[VALIDATION].mean() - voice_passive_margin[VALIDATION].mean())
voice_gap_distance = abs(voice_gap - base_gap)
baseline_sample_distance = float((voice_validation - base_validation).abs().mean())

paired_dod = (
    (activations["voice_active_h"][DISCOVERY, BEST_LAYER] - activations["voice_passive_h"][DISCOVERY, BEST_LAYER])
    - (activations["base_active_h"][DISCOVERY, BEST_LAYER] - activations["base_passive_h"][DISCOVERY, BEST_LAYER])
)
residual_direction = F.normalize(paired_dod.mean(0), dim=0)
residual_center = torch.cat([
    activations["voice_active_h"][DISCOVERY, BEST_LAYER],
    activations["voice_passive_h"][DISCOVERY, BEST_LAYER],
]).mean(0)

def summarize(name, active_margin, passive_margin, features=None):
    combined = torch.cat([active_margin, passive_margin])
    gap_after = float(active_margin.mean() - passive_margin.mean())
    sample_distance = float((combined - base_validation).abs().mean())
    return {
        "arm": name,
        "features": features,
        "voice_gap_before": voice_gap,
        "voice_gap_after": gap_after,
        "gap_reduction": voice_gap - gap_after,
        "gap_recovery_toward_base": 1.0 - abs(gap_after - base_gap) / max(voice_gap_distance, 1e-8),
        "mean_logit_distance_to_base_before": baseline_sample_distance,
        "mean_logit_distance_to_base_after": sample_distance,
        "logit_recovery_toward_base": 1.0 - sample_distance / max(baseline_sample_distance, 1e-8),
        "fixed_voice_rule_accuracy": float(torch.cat([active_margin > 0, passive_margin < 0]).float().mean()),
        "label_changes": int((combined.gt(0) != voice_validation.gt(0)).sum()),
    }

group_results = [summarize(
    "unablated_voice", voice_active_margin[VALIDATION], voice_passive_margin[VALIDATION]
)]
print("Running exact layer-19 residual projection...")
projected_active = evaluate_intervention(
    validation_active_texts, direction=residual_direction, center=residual_center
)
projected_passive = evaluate_intervention(
    validation_passive_texts, direction=residual_direction, center=residual_center
)
group_results.append(summarize("exact_residual_projection", projected_active, projected_passive))

for name, features in FEATURE_GROUPS.items():
    print(f"Running {name}: {features}")
    active_margin = evaluate_intervention(validation_active_texts, features=features)
    passive_margin = evaluate_intervention(validation_passive_texts, features=features)
    group_results.append(summarize(name, active_margin, passive_margin, features))

(OUT / "group_ablation.json").write_text(json.dumps(group_results, indent=2))
for row in group_results:
    print(
        f"{row['arm']:26s} gap={row['voice_gap_after']:8.3f} "
        f"gap_recovery={row['gap_recovery_toward_base']:7.3f} "
        f"sample_recovery={row['logit_recovery_toward_base']:7.3f} "
        f"rule_acc={row['fixed_voice_rule_accuracy']:.3f} flips={row['label_changes']}"
    )

In [ ]:
# Diagnose whether voice dependence was erased or predictions merely collapsed.
def rule_diagnostics(active_margin, passive_margin):
    active_predicts_one = active_margin > 0
    passive_predicts_one = passive_margin > 0
    pair_categories = {
        "active_1_passive_0": int((active_predicts_one & ~passive_predicts_one).sum()),
        "active_0_passive_1": int((~active_predicts_one & passive_predicts_one).sum()),
        "both_1": int((active_predicts_one & passive_predicts_one).sum()),
        "both_0": int((~active_predicts_one & ~passive_predicts_one).sum()),
    }
    return {
        "active_predicts_1_rate": float(active_predicts_one.float().mean()),
        "passive_predicts_1_rate": float(passive_predicts_one.float().mean()),
        "binary_voice_effect": float(active_predicts_one.float().mean() - passive_predicts_one.float().mean()),
        "mean_logit_voice_effect": float(active_margin.mean() - passive_margin.mean()),
        "paired_prediction_agreement": float((active_predicts_one == passive_predicts_one).float().mean()),
        "fixed_voice_rule_accuracy": float(torch.cat([active_predicts_one, ~passive_predicts_one]).float().mean()),
        "pair_categories": pair_categories,
    }

erasure_diagnostics = {
    "unadapted_base": rule_diagnostics(base_active_margin[VALIDATION], base_passive_margin[VALIDATION]),
    "unablated_voice": rule_diagnostics(voice_active_margin[VALIDATION], voice_passive_margin[VALIDATION]),
    "exact_residual_projection": rule_diagnostics(projected_active, projected_passive),
    "interpretation": {
        "erasure_evidence": "Binary and mean-logit voice effects near zero, with high matched-pair prediction agreement.",
        "collapse_warning": "If both_0 or both_1 dominates, voice dependence was erased by label collapse rather than restoration of base behavior.",
    },
}
base_active_prediction = base_active_margin[VALIDATION] > 0
base_passive_prediction = base_passive_margin[VALIDATION] > 0
erasure_diagnostics["exact_residual_projection"]["prediction_agreement_with_base"] = float(torch.cat([
    (projected_active > 0) == base_active_prediction,
    (projected_passive > 0) == base_passive_prediction,
]).float().mean())
(OUT / "rule_erasure_diagnostics.json").write_text(json.dumps(erasure_diagnostics, indent=2))
for arm, values in erasure_diagnostics.items():
    if arm == "interpretation":
        continue
    print(f"\n{arm}")
    print(json.dumps(values, indent=2))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = [row["arm"].replace("_", "\n") for row in group_results]
x = np.arange(len(group_results))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)
axes[0].bar(x, [row["voice_gap_after"] for row in group_results], color="tab:blue")
axes[0].axhline(base_gap, color="black", linestyle="--", linewidth=1.5, label="Unadapted base")
axes[0].axhline(voice_gap, color="tab:red", linestyle=":", linewidth=1.5, label="Voice adapter")
axes[0].set(title="Answer-logit gap after intervention", ylabel="mean logit(1) − logit(0) gap", xticks=x, xticklabels=labels)
axes[0].legend(frameon=False)

width = 0.38
axes[1].bar(x - width / 2, [row["gap_recovery_toward_base"] for row in group_results], width, label="Aggregate-gap recovery", color="tab:green")
axes[1].bar(x + width / 2, [row["logit_recovery_toward_base"] for row in group_results], width, label="Per-example recovery", color="tab:purple")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="Recovery toward unadapted behavior", ylabel="fraction recovered", xticks=x, xticklabels=labels)
axes[1].legend(frameon=False)
for axis in axes:
    axis.tick_params(axis="x", labelrotation=25, labelsize=8)
fig.suptitle(f"Fixed-CoT voice intervention at layer {BEST_LAYER}", fontsize=14)
fig.savefig(OUT / "group_ablation.png", dpi=300, bbox_inches="tight")
plt.show()

run_metadata = {
    "base_model": BASE_MODEL,
    "voice_adapter_sha256": VOICE_SHA256,
    "reused_search_archive": zip_names[0],
    "best_layer": BEST_LAYER,
    "feature_groups": FEATURE_GROUPS,
    "primary_arm": "exact_residual_projection",
    "selection_note": "Groups reuse the prior single-feature validation screen and are exploratory.",
}
(OUT / "experiment.json").write_text(json.dumps(run_metadata, indent=2))
archive = shutil.make_archive("/content/voice_feature_group_ablation_05b", "zip", root_dir=OUT)
files.download(archive)
print("Downloaded", archive)